# VASP Planner V3 Gemma 4 E2B Grounded Continued Training (Unsloth QLoRA)

This notebook continues training from the existing tuned Planner V3 adapter zip:

`/content/drive/MyDrive/planner_v3_e2b_unsloth_qlora.zip`

It trains fully on the refreshed grounded Planner V3 examples generated from `transcripts.md` + `media_skill.md`, using the same runtime prompt format as `planner_prompt.txt`.

It saves a new adapter zip to Drive:

`/content/drive/MyDrive/planner_v3_e2b_unsloth_qlora_grounded_full_v1.zip`

The original adapter zip is not overwritten.


In [16]:
# 1) Check GPU runtime
!nvidia-smi


Tue May 12 21:26:51 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA RTX PRO 6000 Blac...    Off |   00000000:05:00.0 Off |                    0 |
| N/A   31C    P0             88W /  600W |   18132MiB /  97887MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

In [17]:
# 2) Clone/pull repo
REPO_URL = "https://github.com/theextraordinary/VASP.git"
REPO_DIR = "/content/VASP"
import os
if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
else:
    %cd {REPO_DIR}
    !git fetch origin
    !git reset --hard origin/main
%cd {REPO_DIR}



/content/VASP
HEAD is now at 8be588e Created a new v3 planner better tuning
/content/VASP


In [18]:
# 3) Optional: sync repo if needed
# Leave this disabled by default so Colab does not erase local/not-yet-pushed dataset changes.
# Uncomment only if you know origin/main has the refreshed Planner V3 dataset.
!git fetch origin
!git reset --hard origin/main


HEAD is now at 8be588e Created a new v3 planner better tuning


In [19]:
# 4) Install dependencies (Unsloth + trainer stack)
!pip -q install -U unsloth datasets trl peft accelerate bitsandbytes huggingface_hub


In [20]:
# 5) Hugging Face login (needed for gated Gemma checkpoints)
from huggingface_hub import login
login()


In [21]:
# 6) Config
from pathlib import Path

# Replace with your exact Gemma 4 E2B checkpoint if different.
MODEL_NAME = "google/gemma-4-e2b-it"

DATASET_PATH = Path("vasp/a2v/finetuning/planner_v3_dataset/output/planner_v3_500_examples.jsonl")

# Existing tuned adapter in Drive. This is the starting point for second-stage tuning.
PREVIOUS_ADAPTER_ZIP = Path("/content/drive/MyDrive/planner_v3_e2b_unsloth_qlora.zip")
PREVIOUS_ADAPTER_UNZIP_ROOT = Path("/content/adapters/planner_v3_previous")

# New grounded continued-training output. Keep this distinct so the old adapter remains safe.
RUN_NAME = "planner_v3_e2b_unsloth_qlora_grounded_full_v1"
OUTPUT_DIR = Path("vasp/a2v/finetuning/planner_v3_dataset/output") / RUN_NAME
ZIP_NAME = RUN_NAME + ".zip"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

MAX_SEQ_LENGTH = 4096
SEED = 42
FULL_TRAIN = True
VAL_RATIO = 0.0
NUM_TRAIN_EPOCHS = 4
LEARNING_RATE = 3e-5


In [22]:
# 7) Verify dataset and previous adapter zip
from google.colab import drive
import shutil

drive.mount('/content/drive')

if not DATASET_PATH.exists():
    raise FileNotFoundError(f"Missing dataset: {DATASET_PATH}")
if not PREVIOUS_ADAPTER_ZIP.exists():
    raise FileNotFoundError(
        f"Missing previous adapter zip: {PREVIOUS_ADAPTER_ZIP}\n"
        "Upload planner_v3_e2b_unsloth_qlora.zip to /content/drive/MyDrive first."
    )

if PREVIOUS_ADAPTER_UNZIP_ROOT.exists():
    shutil.rmtree(PREVIOUS_ADAPTER_UNZIP_ROOT)
PREVIOUS_ADAPTER_UNZIP_ROOT.mkdir(parents=True, exist_ok=True)

!unzip -q -o "{PREVIOUS_ADAPTER_ZIP}" -d "{PREVIOUS_ADAPTER_UNZIP_ROOT}"

def find_adapter_dir(root: Path) -> Path:
    candidates = sorted(root.rglob("adapter_config.json"))
    if not candidates:
        raise FileNotFoundError(f"No adapter_config.json found after unzipping {PREVIOUS_ADAPTER_ZIP}")
    for cfg in candidates:
        parent = cfg.parent
        if (parent / "adapter_model.safetensors").exists() or (parent / "adapter_model.bin").exists():
            return parent
    return candidates[0].parent

PREVIOUS_ADAPTER_DIR = find_adapter_dir(PREVIOUS_ADAPTER_UNZIP_ROOT)
print("Found dataset:", DATASET_PATH)
print("Found previous adapter zip:", PREVIOUS_ADAPTER_ZIP)
print("Previous adapter dir:", PREVIOUS_ADAPTER_DIR)
print("Previous adapter files:", sorted(x.name for x in PREVIOUS_ADAPTER_DIR.iterdir())[:20])


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Found dataset: vasp/a2v/finetuning/planner_v3_dataset/output/planner_v3_500_examples.jsonl
Found previous adapter zip: /content/drive/MyDrive/planner_v3_e2b_unsloth_qlora.zip
Previous adapter dir: /content/adapters/planner_v3_previous/vasp/a2v/finetuning/planner_v3_dataset/output/planner_v3_e2b_unsloth_qlora/adapter
Previous adapter files: ['README.md', 'adapter_config.json', 'adapter_model.safetensors', 'chat_template.jinja', 'processor_config.json', 'tokenizer.json', 'tokenizer_config.json']


In [23]:
# 8) Load dataset
import json, random
from datasets import Dataset

random.seed(SEED)
rows = [json.loads(x) for x in DATASET_PATH.read_text(encoding="utf-8").splitlines() if x.strip()]
random.shuffle(rows)

if FULL_TRAIN:
    train_rows = rows
    val_rows = []
else:
    val_n = max(10, int(len(rows) * VAL_RATIO))
    val_rows = rows[:val_n]
    train_rows = rows[val_n:]

train_ds = Dataset.from_list(train_rows)
val_ds = Dataset.from_list(val_rows) if val_rows else None

print(f"train={len(train_ds)} val={len(val_ds) if val_ds is not None else 0} full_train={FULL_TRAIN}")


train=500 val=0 full_train=True


In [24]:
# 9) Load model + tokenizer with Unsloth
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=MAX_SEQ_LENGTH,
    dtype=None,
    load_in_4bit=True,
)

if getattr(tokenizer, "pad_token", None) is None and hasattr(tokenizer, "eos_token"):
    tokenizer.pad_token = tokenizer.eos_token
if hasattr(tokenizer, "padding_side"):
    tokenizer.padding_side = "left"


==((====))==  Unsloth 2026.5.2: Fast Gemma4 patching. Transformers: 5.5.0.
   \\   /|    NVIDIA RTX PRO 6000 Blackwell Server Edition. Num GPUs = 1. Max memory: 94.971 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 12.0. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/2011 [00:00<?, ?it/s]

In [25]:
# 10) Convert chat rows to text via chat template
def to_text(example):
    text = tokenizer.apply_chat_template(
        example["messages"],
        tokenize=False,
        add_generation_prompt=False,
    )
    return {"text": text}

train_ds = train_ds.map(to_text)
if val_ds is not None:
    val_ds = val_ds.map(to_text)
print(train_ds[0]["text"][:1000])


Map:   0%|          | 0/500 [00:00<?, ? examples/s]

<bos><|turn>user
You are Media-Transcript Matching Planner.
Your job is to match media assets to the transcript text parts they explain best.

Rules:
- Return exactly one complete valid JSON object only. Do not repeat the JSON. Do not add markdown or extra text.
- Use all mandatory media at least once.
- Optional media may be used only if it strongly matches transcript text.
- Mandatory media has priority over optional media.
- Each text part may be matched to only ONE media.
- One media may match multiple text parts.
- Prefer matching every meaningful transcript part; avoid leaving text unmatched unless no media fits.
- Match can be a word, phrase, sentence fragment, or full sentence.
- Always prioritize aim over about.
- If aim explicitly says media should be used for a certain topic/text, obey aim first.
- Match creatively like a video editor: use media where it improves clarity, humor, emotional impact, or topic relevance.
- Do not invent media ids.
- Do not output timestamps; segm

In [26]:
# 11) Load previous tuned adapter and keep it trainable
from peft import PeftModel


def unwrap_gemma4_clippable_linear(module, prefix=""):
    """PEFT cannot inject LoRA into Gemma4ClippableLinear wrappers.

    The trained adapter targets names like q_proj/k_proj/v_proj, but in Gemma 4
    those modules can be wrapper objects containing an inner .linear layer.
    Replacing the wrapper with its inner torch.nn.Linear keeps the same module
    name, so adapter keys still match while PEFT sees a supported target type.
    """
    replaced = []
    for name, child in list(module.named_children()):
        child_path = f"{prefix}.{name}" if prefix else name
        if child.__class__.__name__ == "Gemma4ClippableLinear" and hasattr(child, "linear"):
            setattr(module, name, child.linear)
            replaced.append(child_path)
            continue
        replaced.extend(unwrap_gemma4_clippable_linear(child, child_path))
    return replaced

unwrapped_modules = unwrap_gemma4_clippable_linear(model)
print(f"Unwrapped Gemma4ClippableLinear modules for PEFT: {len(unwrapped_modules)}")
if unwrapped_modules:
    print("First unwrapped modules:", unwrapped_modules[:12])

model = PeftModel.from_pretrained(
    model,
    str(PREVIOUS_ADAPTER_DIR),
    adapter_name="planner_v3",
    is_trainable=True,
)
model.train()
model.print_trainable_parameters()
print("Continuing training from:", PREVIOUS_ADAPTER_DIR)


Unwrapped Gemma4ClippableLinear modules for PEFT: 232
First unwrapped modules: ['model.vision_tower.encoder.layers.0.self_attn.q_proj', 'model.vision_tower.encoder.layers.0.self_attn.k_proj', 'model.vision_tower.encoder.layers.0.self_attn.v_proj', 'model.vision_tower.encoder.layers.0.self_attn.o_proj', 'model.vision_tower.encoder.layers.0.mlp.gate_proj', 'model.vision_tower.encoder.layers.0.mlp.up_proj', 'model.vision_tower.encoder.layers.0.mlp.down_proj', 'model.vision_tower.encoder.layers.1.self_attn.q_proj', 'model.vision_tower.encoder.layers.1.self_attn.k_proj', 'model.vision_tower.encoder.layers.1.self_attn.v_proj', 'model.vision_tower.encoder.layers.1.self_attn.o_proj', 'model.vision_tower.encoder.layers.1.mlp.gate_proj']
trainable params: 62,078,976 || all params: 5,185,256,992 || trainable%: 1.1972
Continuing training from: /content/adapters/planner_v3_previous/vasp/a2v/finetuning/planner_v3_dataset/output/planner_v3_e2b_unsloth_qlora/adapter


/usr/local/lib/python3.12/dist-packages/peft/peft_model.py:622: UserWarning: Found missing adapter keys while loading the checkpoint: ['base_model.model.model.vision_tower.encoder.layers.0.self_attn.q_proj.lora_A.planner_v3.weight', 'base_model.model.model.vision_tower.encoder.layers.0.self_attn.q_proj.lora_B.planner_v3.weight', 'base_model.model.model.vision_tower.encoder.layers.0.self_attn.k_proj.lora_A.planner_v3.weight', 'base_model.model.model.vision_tower.encoder.layers.0.self_attn.k_proj.lora_B.planner_v3.weight', 'base_model.model.model.vision_tower.encoder.layers.0.self_attn.v_proj.lora_A.planner_v3.weight', 'base_model.model.model.vision_tower.encoder.layers.0.self_attn.v_proj.lora_B.planner_v3.weight', 'base_model.model.model.vision_tower.encoder.layers.0.self_attn.o_proj.lora_A.planner_v3.weight', 'base_model.model.model.vision_tower.encoder.layers.0.self_attn.o_proj.lora_B.planner_v3.weight', 'base_model.model.model.vision_tower.encoder.layers.0.mlp.gate_proj.lora_A.planne

In [27]:
# 12) Fully continue training on grounded Planner V3 examples with TRL SFTTrainer
from trl import SFTTrainer
from transformers import TrainingArguments
import inspect

# Transformers API compatibility: evaluation_strategy vs eval_strategy
_ta_params = inspect.signature(TrainingArguments.__init__).parameters
_strategy_key = "evaluation_strategy" if "evaluation_strategy" in _ta_params else "eval_strategy"

train_kwargs = dict(
    output_dir=str(OUTPUT_DIR / "checkpoints"),
    per_device_train_batch_size=2,
    gradient_accumulation_steps=8,
    num_train_epochs=NUM_TRAIN_EPOCHS,
    learning_rate=LEARNING_RATE,
    warmup_ratio=0.02,
    weight_decay=0.0,
    logging_steps=10,
    save_steps=50,
    save_strategy="steps",
    bf16=True,
    fp16=False,
    optim="adamw_8bit",
    lr_scheduler_type="cosine",
    seed=SEED,
    report_to="none",
)
if val_ds is not None:
    train_kwargs.update(
        per_device_eval_batch_size=2,
        eval_steps=50,
        load_best_model_at_end=True,
    )
    train_kwargs[_strategy_key] = "steps"
else:
    train_kwargs[_strategy_key] = "no"
    train_kwargs["load_best_model_at_end"] = False

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    dataset_text_field="text",
    max_seq_length=MAX_SEQ_LENGTH,
    packing=False,
    args=TrainingArguments(**train_kwargs),
)

train_result = trainer.train()
print(train_result)


warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.
warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Unsloth: Tokenizing ["text"] (num_proc=52):   0%|          | 0/500 [00:00<?, ? examples/s]

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': 2}.
==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 500 | Num Epochs = 4 | Total steps = 128
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 8
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 8 x 1) = 16
 "-____-"     Trainable parameters = 62,078,976 of 5,185,256,992 (1.20% trained)


Step,Training Loss
10,0.244134
20,0.089544
30,0.049713
40,0.048629
50,0.031726
60,0.028674
70,0.034790
80,0.025032
90,0.023932
100,0.030538


Unsloth: Restored added_tokens_decoder metadata in vasp/a2v/finetuning/planner_v3_dataset/output/planner_v3_e2b_unsloth_qlora_grounded_full_v1/checkpoints/checkpoint-50/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in vasp/a2v/finetuning/planner_v3_dataset/output/planner_v3_e2b_unsloth_qlora_grounded_full_v1/checkpoints/checkpoint-100/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in vasp/a2v/finetuning/planner_v3_dataset/output/planner_v3_e2b_unsloth_qlora_grounded_full_v1/checkpoints/checkpoint-128/tokenizer_config.json.


TrainOutput(global_step=128, training_loss=0.05286410066764802, metrics={'train_runtime': 250.1123, 'train_samples_per_second': 7.996, 'train_steps_per_second': 0.512, 'total_flos': 2.9906455363584e+16, 'train_loss': 0.05286410066764802, 'epoch': 4.0})


In [28]:
# 13) Save continued adapter + tokenizer
adapter_dir = OUTPUT_DIR / "adapter"
trainer.model.save_pretrained(str(adapter_dir))
tokenizer.save_pretrained(str(adapter_dir))
print("Saved continued adapter:", adapter_dir)


Unsloth: Restored added_tokens_decoder metadata in vasp/a2v/finetuning/planner_v3_dataset/output/planner_v3_e2b_unsloth_qlora_grounded_full_v1/adapter/tokenizer_config.json.


Saved continued adapter: vasp/a2v/finetuning/planner_v3_dataset/output/planner_v3_e2b_unsloth_qlora_grounded_full_v1/adapter


In [29]:
# 14) Zip continued adapter
!cd /content/VASP && rm -f {ZIP_NAME} && zip -r {ZIP_NAME} {adapter_dir}


  adding: vasp/a2v/finetuning/planner_v3_dataset/output/planner_v3_e2b_unsloth_qlora_grounded_full_v1/adapter/ (stored 0%)
  adding: vasp/a2v/finetuning/planner_v3_dataset/output/planner_v3_e2b_unsloth_qlora_grounded_full_v1/adapter/planner_v3/ (stored 0%)
  adding: vasp/a2v/finetuning/planner_v3_dataset/output/planner_v3_e2b_unsloth_qlora_grounded_full_v1/adapter/planner_v3/adapter_model.safetensors (deflated 23%)
  adding: vasp/a2v/finetuning/planner_v3_dataset/output/planner_v3_e2b_unsloth_qlora_grounded_full_v1/adapter/planner_v3/adapter_config.json (deflated 58%)
  adding: vasp/a2v/finetuning/planner_v3_dataset/output/planner_v3_e2b_unsloth_qlora_grounded_full_v1/adapter/README.md (deflated 65%)
  adding: vasp/a2v/finetuning/planner_v3_dataset/output/planner_v3_e2b_unsloth_qlora_grounded_full_v1/adapter/tokenizer.json (deflated 83%)
  adding: vasp/a2v/finetuning/planner_v3_dataset/output/planner_v3_e2b_unsloth_qlora_grounded_full_v1/adapter/processor_config.json (deflated 69%)
  a

In [30]:
# 15) Save continued zip to Google Drive
from google.colab import drive
drive.mount('/content/drive')
!cp /content/VASP/{ZIP_NAME} /content/drive/MyDrive/
print("Saved grounded continued adapter to Drive:", "/content/drive/MyDrive/" + ZIP_NAME)


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Saved grounded continued adapter to Drive: /content/drive/MyDrive/planner_v3_e2b_unsloth_qlora_grounded_full_v1.zip


In [31]:
# 16) Download continued zip to local machine
from google.colab import files
files.download("/content/VASP/" + ZIP_NAME)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>